In [1]:
import pandas as pd

df_raw = pd.read_pickle("processed_data/dataset_raw_datpacket_user.pkl")
print(len(df_raw))

101876


In [2]:
df_raw.keys()

Index(['Object', 'Time Step', 'Id', 'User', 'Application', 'Size', 'Status',
       'Queue Delay', 'Transmission Delay', 'Processing Delay',
       'Propagation Delay', 'Total Delay', 'Total Path', 'Hops', 'Object_user',
       'Instance ID', 'Coordinates', 'Base Station', 'Delays', 'Delay SLAs',
       'Communication Paths', 'Making Requests', 'Access History',
       'Distribution', 'Scenario', 'App_ms', 'Run'],
      dtype='str')

In [3]:
columns_to_keep = [

    'Distribution', 'Scenario', 'App_ms', 'Run', 'Time Step', 'Object', 'Id',

    # DataPacket
    'Status',
    'User',
    'Application',
    'Processing Delay',
    'Propagation Delay',
    'Total Delay',
    'Total Path',
    'Hops',

    # User
    'Delays', 'Delay SLAs',

]

df_light = df_raw[columns_to_keep].copy()

In [4]:
user_data_SLA = df_light.groupby(['Scenario', 'App_ms'])['Delay SLAs'].first().reset_index()
user_data_SLA['Delay SLAs'] = user_data_SLA['Delay SLAs'].apply(lambda d: list(d.values())[0])
user_data_SLA

,Scenario,App_ms,Delay SLAs
0,1_26_solution_v0,1SMM,63.486
1,1_26_solution_v0,1SMS,63.486
2,1_26_solution_v0,1SSM,63.486
3,1_26_solution_v0,1SSS,63.486
4,1_26_solution_v10,1MMM,69.112
5,1_26_solution_v10,1MMS,69.112
6,1_26_solution_v10,1MSM,69.112
7,1_26_solution_v10,1MSS,69.112
8,1_26_solution_v10,1SMM,69.112
9,1_26_solution_v10,1SMS,69.112


In [5]:
df_packets = df_light[df_light["Status"] == "finished"].copy()

if 'Delay SLAs' in df_packets.columns:
    df_packets = df_packets.drop(columns=['Delay SLAs'])

df_packets_sla = df_packets.merge(user_data_SLA, on=['Scenario', 'App_ms'], how='left')

columns_to_keep = [

    'Distribution', 'Scenario', 'App_ms', 'Run', 'Time Step', 'Object', 'Id',

    # DataPacket
    'Status',
    'User',
    'Application',
    'Processing Delay',
    'Propagation Delay',
    'Total Delay',
    'Delay SLAs', # aggiunto da user 
    'Total Path',
    'Hops',
]

df_packets_sla = df_packets_sla[columns_to_keep].copy()

print("Duplicate:", df_packets_sla.duplicated(subset=['Distribution', 'Scenario', 'App_ms', 'Run']).sum())

Duplicate: 6904


In [6]:
idx_max = df_packets_sla.groupby(['Distribution', 'Scenario', 'App_ms', 'Run'])['Total Delay'].idxmax()

df_worst_case = df_packets_sla.loc[idx_max].reset_index(drop=True).copy()

print("Duplicate:", df_worst_case.duplicated(subset=['Distribution', 'Scenario', 'App_ms', 'Run']).sum())

Duplicate: 0


In [10]:
df_worst_case['SLA_Violation'] = df_worst_case['Total Delay'] > df_worst_case['Delay SLAs']

df_worst_case['SLA_Margin_Perc'] = ((df_worst_case['Total Delay'] - df_worst_case['Delay SLAs']) / df_worst_case['Delay SLAs'] * 100).round(3)

df_worst_case['SLA_Margin'] = df_worst_case['Total Delay'] - df_worst_case['Delay SLAs']
df_worst_case['Violation_Amount'] = df_worst_case['SLA_Margin'].clip(lower=0)

cols_final = [
    'Distribution', 'Scenario', 'App_ms', 'Run', 'User', 'Application',
    'Total Delay', 'Delay SLAs', 'SLA_Violation', 'SLA_Margin_Perc',
    'SLA_Margin', 'Violation_Amount',
    'Processing Delay', 'Propagation Delay', 'Hops'
]

df_worst_case[cols_final].to_pickle("processed_data/dataset_worst_case_datapacket_user.pkl")

df_worst = pd.read_pickle("processed_data/dataset_worst_case_datapacket_user.pkl")
df_worst.head(11)

,Distribution,Scenario,App_ms,Run,User,Application,Total Delay,Delay SLAs,SLA_Violation,SLA_Margin_Perc,SLA_Margin,Violation_Amount,Processing Delay,Propagation Delay,Hops
0,exponential,1_26_solution_v0,1SMM,run_0,1,1,72,63.486,True,13.411,8.514,8.514,45,27,"[{'hop_index': 0, 'link_index': 0, 'source': 1..."
1,exponential,1_26_solution_v0,1SMM,run_1,1,1,37,63.486,False,-41.719,-26.486,0.000,10,27,"[{'hop_index': 0, 'link_index': 0, 'source': 1..."
2,exponential,1_26_solution_v0,1SMM,run_2,1,1,49,63.486,False,-22.818,-14.486,0.000,22,27,"[{'hop_index': 0, 'link_index': 0, 'source': 1..."
3,exponential,1_26_solution_v0,1SMM,run_3,1,1,52,63.486,False,-18.092,-11.486,0.000,25,27,"[{'hop_index': 0, 'link_index': 0, 'source': 1..."
4,exponential,1_26_solution_v0,1SMM,run_4,1,1,48,63.486,False,-24.393,-15.486,0.000,21,27,"[{'hop_index': 0, 'link_index': 0, 'source': 1..."
5,exponential,1_26_solution_v0,1SMM,run_5,1,1,50,63.486,False,-21.242,-13.486,0.000,23,27,"[{'hop_index': 0, 'link_index': 0, 'source': 1..."
6,exponential,1_26_solution_v0,1SMM,run_6,1,1,58,63.486,False,-8.641,-5.486,0.000,31,27,"[{'hop_index': 0, 'link_index': 0, 'source': 1..."
7,exponential,1_26_solution_v0,1SMM,run_7,1,1,44,63.486,False,-30.693,-19.486,0.000,17,27,"[{'hop_index': 0, 'link_index': 0, 'source': 1..."
8,exponential,1_26_solution_v0,1SMM,run_8,1,1,52,63.486,False,-18.092,-11.486,0.000,25,27,"[{'hop_index': 0, 'link_index': 0, 'source': 1..."
9,exponential,1_26_solution_v0,1SMM,run_9,1,1,55,63.486,False,-13.367,-8.486,0.000,28,27,"[{'hop_index': 0, 'link_index': 0, 'source': 1..."
